In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("akrgbh/deepfake2")

print("Path to dataset files:", path)

In [ ]:
# Define the directory paths for the training, validation, and test datasets
# These paths point to folders on Google Drive where the image data is stored
train_dir ='/kaggle/input/datasets/akrgbh/deepfake2/Dataset/Train'
val_dir = '/kaggle/input/datasets/akrgbh/deepfake2/Dataset/Validation'
test_dir = '/kaggle/input/datasets/akrgbh/deepfake2/Dataset/Test'

In [ ]:
# Import necessary modules and libraries for data handling and visualization
import os # For interacting with the operating system, e.g., listing directory contents
import random # For generating random numbers, useful for selecting random images
import cv2 # OpenCV library for image processing tasks (e.g., filtering)
import numpy as np # NumPy library for numerical operations, especially with arrays
import matplotlib.pyplot as plt # Matplotlib for creating static, interactive, and animated visualizations
import torch # Core PyTorch library
import torch.nn as nn # Neural network module for defining layers
from torch.utils.data import DataLoader # Utility for loading data in batches
from torchvision import datasets, transforms # Datasets and transformations for computer vision
import timm # PyTorch Image Models library

In [ ]:
# Print the installed PyTorch version to verify the environment setup
print("PyTorch version:", torch.__version__)
# Check if a CUDA-enabled GPU is available for use to leverage GPU acceleration
print("CUDA available:", torch.cuda.is_available())

# Conditional statement to inform the user about GPU status
if torch.cuda.is_available():
    # If GPU is available, print the name of the first detected GPU for confirmation
    print("GPU:", torch.cuda.get_device_name(0))
else:
    # If no GPU is found, advise the user to enable it in Colab runtime settings
    print("⚠️ GPU is not enabled. Go to Runtime → Change runtime type → GPU")

In [ ]:
# Define the device to be used for computations: 'cuda' if a GPU is available, otherwise 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Print the selected device to confirm whether GPU or CPU is being utilized
print(f"Using device: {device}")

In [ ]:
# Check and print whether each directory path actually exists
print("Train exists:", os.path.exists(train_dir))
print("Validation exists:", os.path.exists(val_dir))
print("Test exists:", os.path.exists(test_dir))

In [ ]:
# Import the 'os' module again
import os


# List all subdirectories within the training directory, which represent the classes
classes = os.listdir(train_dir)
# Get the number of classes based on the length of the list
num_classes = len(classes)

# Display the total number of classes found
print(f'Number of Classes: {num_classes}')

# List and print the names of subfolders (classes) within the training directory
print("Train folders:", os.listdir(train_dir))
# List and print the names of subfolders (classes) within the validation directory
print("Validation folders:", os.listdir(val_dir))
# List and print the names of subfolders (classes) within the test directory
print("Test folders:", os.listdir(test_dir))

In [ ]:
# Choose a random class (e.g., 'Fake' or 'Real') from the 'classes' list

import os

# The 'classes' variable is already defined from a previous cell (mvB2WLikZ1HP)
# classes = train_dataset.classes # This line caused the NameError

random_class = random.choice(classes)

# Construct the full path to the directory of the randomly chosen class
random_class_path = os.path.join(train_dir, random_class)

# Get a list of all image file names present in that specific class directory
image_files = os.listdir(random_class_path)

# Choose a random image file name from the list of images in the chosen class
random_img_name = random.choice(image_files)

# Construct the complete file path for the randomly selected image
random_img_path = os.path.join(random_class_path, random_img_name)

# Read the image file using Matplotlib's imread function (reads as a NumPy array)
random_img = plt.imread(random_img_path)

# Create a new figure for displaying the image with a specified size
plt.figure(figsize=(6, 6))
# Display the image
plt.imshow(random_img)
# Set the title of the plot to indicate the class and image shape
plt.title(f'{random_class} class\nShape: {random_img.shape}')
# Turn off the axis ticks and labels for a cleaner image display
plt.axis('off')
# Show the plot
plt.show()

In [ ]:
# Import DataLoader from PyTorch's data utility for efficient batch loading
from torch.utils.data import DataLoader

# Define image size and batch size for PyTorch DataLoader
IMG_SIZE = 224 # Target size (width and height) for images
BATCH_SIZE = 16 # Number of images per batch for PyTorch DataLoader

# Define transformations for the training dataset
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), # Resize images to the specified IMG_SIZE
    transforms.RandomHorizontalFlip(), # Randomly flip images horizontally for data augmentation
    transforms.RandomRotation(10), # Randomly rotate images by up to 10 degrees
    transforms.ColorJitter( # Randomly change brightness and contrast
        brightness=0.2, # Adjust brightness by up to 20%
        contrast=0.2 # Adjust contrast by up to 20%
    ),
    transforms.ToTensor(), # Convert images to PyTorch Tensors
    transforms.Normalize( # Normalize pixel values using mean and standard deviation
        mean=[0.485, 0.456, 0.406], # Mean values for R, G, B channels (ImageNet statistics)
        std=[0.229, 0.224, 0.225] # Standard deviation values for R, G, B channels (ImageNet statistics)
    )
])

# Define transformations for the validation and test datasets (no augmentation)
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), # Resize images to the specified IMG_SIZE
    transforms.ToTensor(), # Convert images to PyTorch Tensors
    transforms.Normalize( # Normalize pixel values using mean and standard deviation
        mean=[0.485, 0.456, 0.406], # Mean values for R, G, B channels (ImageNet statistics)
        std=[0.229, 0.224, 0.225] # Standard deviation values for R, G, B channels (ImageNet statistics)
    )
])
# Create ImageFolder datasets for training, validation, and test sets
# ImageFolder expects images sorted into subfolders named after their class
train_dataset = datasets.ImageFolder(
    train_dir, # Path to the training data directory
    transform=train_transform # Apply the defined training transformations
)

val_dataset = datasets.ImageFolder(
    val_dir, # Path to the validation data directory
    transform=val_test_transform # Apply the defined validation/test transformations
)

test_dataset = datasets.ImageFolder(
    test_dir, # Path to the test data directory
    transform=val_test_transform # Apply the defined validation/test transformations
)

# Create a DataLoader for the training dataset
train_loader = DataLoader(
    train_dataset, # The dataset to load data from
    batch_size=BATCH_SIZE, # Number of samples per batch
    shuffle=True, # Shuffle the data every epoch for better training
    num_workers=0, # Set to 0 to load data in the main process
    pin_memory=True # Copy Tensors to CUDA pinned memory before returning them
)

# Create a DataLoader for the validation dataset
val_loader = DataLoader(
    val_dataset, # The dataset to load data from
    batch_size=BATCH_SIZE, # Number of samples per batch
    shuffle=False, # No need to shuffle validation data
    num_workers=0, # Set to 0 to load data in the main process
    pin_memory=True # Copy Tensors to CUDA pinned memory
)

# Create a DataLoader for the test dataset
test_loader = DataLoader(
    test_dataset, # The dataset to load data from
    batch_size=BATCH_SIZE, # Number of samples per batch
    shuffle=False, # No need to shuffle test data
    num_workers=0, # Set to 0 to load data in the main process
    pin_memory=True # Copy Tensors to CUDA pinned memory
)

# Print the class names inferred by ImageFolder
print("Classes:", train_dataset.classes)
# Print the total number of images in each dataset
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

# Print the number of batches in each DataLoader
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# Apply a bilateral filter for noise reduction while preserving edges of the 'random_img'
# d: Diameter of each pixel neighborhood that is used during filtering. A larger value means more pixels are considered.
# sigmaColor: Filter sigma in the color space. A larger value means farther colors within the neighborhood will be mixed.
# sigmaSpace: Filter sigma in the coordinate space. A larger value means farther pixels will influence each other if their colors are close enough.
filtered_img = cv2.bilateralFilter(random_img, d=9, sigmaColor=100, sigmaSpace=100)

# Create a figure to display both the original and filtered images side-by-side
plt.figure(figsize=(12, 6))

# Create the first subplot for the original image
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.imshow(random_img) # Display the original image
plt.title('Original Image') # Set title for the original image plot
plt.axis('off') # Turn off axis ticks and labels

# Create the second subplot for the bilateral filtered image
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.imshow(filtered_img) # Display the filtered image
plt.title('Bilateral Filtered Image') # Set title for the filtered image plot
plt.axis('off') # Turn off axis ticks and labels

# Show the combined plot
plt.show()

In [ ]:
# Create a Swin Transformer model with a tiny architecture
# 'swin_tiny_patch4_window7_224': Specifies the model architecture and input resolution
# pretrained=True: Loads pre-trained weights (usually from ImageNet) for better performance and faster convergence
# num_classes=0: Initializes the model without a classification head, suitable for feature extraction
swin_model = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=2 # Set to 0 to get features without the classification head
)

# Print a success message once the Swin Transformer model is loaded to indicate readiness
print("Swin Transformer model loaded successfully!")

In [ ]:
swin_model.eval()
swin_model = swin_model.to(device)

images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    output = swin_model(images)

print("Input:", images.shape)
print("Output:", output.shape)

In [ ]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    swin_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Loss function and optimizer ready")

In [ ]:
num_epochs = 50  # Increased number of epochs

history_swin = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print("Starting training with increased epochs...")

for epoch in range(num_epochs):  # Training Phase
    swin_model.train()  # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)  # Move data to the specified device

        optimizer.zero_grad()  # Zero the gradients from the previous iteration
        outputs = swin_model(images)  # Perform a forward pass
        loss = criterion(outputs, labels)  # Calculate the loss
        loss.backward()  # Perform backpropagation
        optimizer.step()  # Update model parameters

        running_loss += loss.item() * images.size(0)  # Accumulate training loss
        _, predicted = torch.max(outputs.data, 1)  # Get the predicted classes
        total_samples += labels.size(0)  # Update total samples count
        correct_predictions += (predicted == labels).sum().item()  # Update correct predictions count

    # Calculate epoch-wise training loss and accuracy
    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_acc = correct_predictions / total_samples
    history_swin['train_loss'].append(epoch_train_loss)
    history_swin['train_acc'].append(epoch_train_acc)

    # Validation Phase
    swin_model.eval()  # Set the model to evaluation mode
    val_running_loss = 0.0
    val_correct_predictions = 0
    val_total_samples = 0

    with torch.no_grad():  # Disable gradient calculation for validation
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)  # Move data to the specified device

            outputs = swin_model(images)  # Perform a forward pass
            loss = criterion(outputs, labels)  # Calculate the loss

            val_running_loss += loss.item() * images.size(0)  # Accumulate validation loss
            _, predicted = torch.max(outputs.data, 1)  # Get the predicted classes
            val_total_samples += labels.size(0)  # Update total samples count
            val_correct_predictions += (predicted == labels).sum().item()  # Update correct predictions count

    # Calculate epoch-wise validation loss and accuracy
    epoch_val_loss = val_running_loss / len(val_dataset)
    epoch_val_acc = val_correct_predictions / val_total_samples
    history_swin['val_loss'].append(epoch_val_loss)
    history_swin['val_acc'].append(epoch_val_acc)

    # Print epoch results
    print(f'Epoch {epoch+1}/{num_epochs} - ' +
          f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, ' +
          f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}')

print("Training with increased epochs complete!")

In [ ]:
# Put the model in evaluation mode
swin_model.eval()

all_labels = []
all_predictions = []
all_logits = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = swin_model(images)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())
        all_logits.extend(outputs.cpu().numpy())

print("Labels and predictions and logits for the full test set collected.")

In [ ]:
print("Classes:", train_dataset.classes)

print("Predicted:", torch.argmax(outputs, dim=1).cpu().numpy())
print("Actual:   ", labels.cpu().numpy())

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch

print("Confusion Matrix libraries ready!")

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_dataset.classes)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():  # Disable gradient calculations for inference
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)  # Move data to the specified device

            outputs = model(images)  # Perform a forward pass
            loss = criterion(outputs, labels)  # Calculate the loss

            running_loss += loss.item() * images.size(0)  # Accumulate loss
            _, predicted = torch.max(outputs.data, 1)  # Get the predicted classes
            total_samples += labels.size(0)  # Update total samples count
            correct_predictions += (predicted == labels).sum().item()  # Update correct predictions count

    final_loss = running_loss / total_samples
    final_acc = correct_predictions / total_samples
    return final_loss, final_acc

print("Evaluation function defined!")

In [ ]:
final_test_loss, final_test_acc = evaluate(swin_model, test_loader, criterion, device)
print(f'\nFinal Test Loss: {final_test_loss:.4f}, Final Test Accuracy: {final_test_acc:.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score

# Calculate Accuracy
accuracy = accuracy_score(all_labels, all_predictions)

# Calculate Precision (using 'weighted' for multiclass)
precision = precision_score(all_labels, all_predictions, average='weighted')

# Calculate Recall (using 'weighted' for multiclass)
recall = recall_score(all_labels, all_predictions, average='weighted')

# Calculate F1-score (using 'weighted' for multiclass)
f1 = f1_score(all_labels, all_predictions, average='weighted')

# Sensitivity is the same as Recall for binary classification. For multiclass, it's often interpreted per class or as the weighted average recall.
sensitivity = recall

print(f'\nAccuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall (Sensitivity): {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Convert logits to probabilities
all_probs = torch.softmax(
    torch.tensor(all_logits, dtype=torch.float32), dim=1
).numpy()

# Number of classes
num_classes = len(test_dataset.classes)

# Convert labels to numpy array
all_labels = np.asarray(all_labels)

# Create one-hot encoded labels for ALL classes
binarized_labels = np.zeros((len(all_labels), num_classes))

for i, label in enumerate(all_labels):
    binarized_labels[i, int(label)] = 1

# Store ROC values
fpr = {}
tpr = {}
roc_auc = {}

# Calculate ROC for each class
for i in range(num_classes):

    # Check whether both positive and negative samples exist
    if len(np.unique(binarized_labels[:, i])) < 2:
        print(f"Skipping class {test_dataset.classes[i]}: "
              f"only one label is present in test data.")
        continue

    fpr[i], tpr[i], _ = roc_curve(
        binarized_labels[:, i],
        all_probs[:, i]
    )

    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure(figsize=(10, 8))

for i in roc_auc:
    plt.plot(
        fpr[i],
        tpr[i],
        label=f'{test_dataset.classes[i]} (AUC = {roc_auc[i]:.2f})'
    )

plt.plot(
    [0, 1],
    [0, 1],
    'k--',
    label='Random Classifier'
)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - One-vs-Rest')

plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
print("Number of classes:", num_classes)
print("Dataset classes:", test_dataset.classes)
print("Unique labels:", np.unique(all_labels))
print("all_labels shape:", np.array(all_labels).shape)
print("all_logits shape:", np.array(all_logits).shape)
print("binarized_labels shape:", binarized_labels.shape)
print("all_probs shape:", all_probs.shape)

In [ ]:
import matplotlib.pyplot as plt

# Plotting training & validation loss and accuracy
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot training and validation loss
axes[0].plot(history_swin['train_loss'], label='Train Loss')
axes[0].plot(history_swin['val_loss'], label='Validation Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Plot training and validation accuracy
axes[1].plot(history_swin['train_acc'], label='Train Accuracy')
axes[1].plot(history_swin['val_acc'], label='Validation Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
torch.save(
    swin_model.state_dict(),
    "best_swin_model.pth"
)

print("Model saved successfully!")

In [ ]:
swin_model.load_state_dict(
    torch.load(
        "best_swin_model.pth",
        map_location=device
    )
)

swin_model = swin_model.to(device)
swin_model.eval()

In [ ]:
from PIL import Image
import torch
import matplotlib.pyplot as plt

# Load image
image_to_predict = Image.open(random_img_path).convert("RGB")

# Transform
input_tensor = val_test_transform(image_to_predict)

# Add batch dimension
input_batch = input_tensor.unsqueeze(0).to(device)

# Model
swin_model = swin_model.to(device)
swin_model.eval()

# Prediction
with torch.no_grad():
    output = swin_model(input_batch)
    probabilities = torch.softmax(output, dim=1)

# Get predicted index
predicted_class_idx = torch.argmax(probabilities, dim=1).item()

# Number of classes produced by model
num_classes = output.shape[1]

# Create class names dynamically
class_names = [f"Class {i}" for i in range(num_classes)]

# Get predicted label
predicted_class_label = class_names[predicted_class_idx]

if predicted_class_idx == 0:
    predicted_class_label = "Fake"
else:
    predicted_class_label = "Real"

# Confidence
confidence = probabilities[0, predicted_class_idx].item() * 100

# Display image
plt.figure(figsize=(6, 6))
plt.imshow(image_to_predict)

plt.title(
    f"Prediction: {predicted_class_label}\n"
    f"Confidence: {confidence:.2f}%"
)

plt.axis("off")
plt.show()

# Print result
print("Model output shape:", output.shape)
print("Number of classes:", num_classes)
print("Predicted class index:", predicted_class_idx)

print(
    f"The model predicts this image is: "
    f"{predicted_class_label} "
    f"with {confidence:.2f}% confidence.")

In [ ]:
print("Output shape:", output.shape)
print("Probabilities:", probabilities)
print("Predicted index:", predicted_class_idx)